# 01 - Data Quality and EDA

This notebook audits the raw Kaggle insurance cross-sell dataset, validates the competition contract, and identifies customer segments with materially different response behavior.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

## Dataset Contract

In [2]:
import pandas as pd

from insurance_propensity.config import DATA_RAW_DIR, REPORT_DIR
from insurance_propensity.data.validation import load_raw_datasets, dataset_quality_summary

bundle = load_raw_datasets(DATA_RAW_DIR)
summary = dataset_quality_summary(bundle)
pd.Series(
    {
        "train_rows": summary["train_shape"][0],
        "train_columns": summary["train_shape"][1],
        "test_rows": summary["test_shape"][0],
        "sample_submission_rows": summary["sample_submission_shape"][0],
        "train_missing_values": summary["total_missing_values"]["train"],
        "test_missing_values": summary["total_missing_values"]["test"],
        "duplicate_train_rows": summary["duplicate_rows"]["train"],
        "train_unique_id": summary["id_quality"]["train_unique_id"],
        "test_submission_id_match": summary["id_quality"]["test_submission_id_match"],
    }
).to_frame("value")

,value
train_rows,381109
train_columns,12
test_rows,127037
sample_submission_rows,127037
train_missing_values,0
test_missing_values,0
duplicate_train_rows,0
train_unique_id,True
test_submission_id_match,True


## Response Balance

In [3]:
response_distribution = (
    bundle.train["Response"]
    .value_counts(normalize=True)
    .rename_axis("response")
    .reset_index(name="share")
    .assign(share=lambda frame: frame["share"].round(4))
)
response_distribution

,response,share
0,0,0.8774
1,1,0.1226


## Segment-Level Signal

In [4]:
train = bundle.train
segment_tables = []
for column in ["Gender", "Vehicle_Age", "Vehicle_Damage", "Previously_Insured"]:
    table = (
        train.groupby(column, observed=True)["Response"]
        .agg(customers="size", responders="sum", response_rate="mean")
        .reset_index()
        .rename(columns={column: "segment_value"})
    )
    table.insert(0, "dimension", column)
    segment_tables.append(table)

pd.concat(segment_tables, ignore_index=True).sort_values(
    ["dimension", "response_rate"],
    ascending=[True, False],
)

,dimension,segment_value,customers,responders,response_rate
1,Gender,Male,206089,28525,0.138411
0,Gender,Female,175020,18185,0.103902
7,Previously_Insured,0,206481,46552,0.225454
8,Previously_Insured,1,174628,158,0.000905
4,Vehicle_Age,> 2 Years,16007,4702,0.293746
2,Vehicle_Age,1-2 Year,200316,34806,0.173755
3,Vehicle_Age,< 1 Year,164786,7202,0.043705
6,Vehicle_Damage,Yes,192413,45728,0.237655
5,Vehicle_Damage,No,188696,982,0.005204


## Reusable EDA Output

In [5]:
pd.read_csv(REPORT_DIR / 'eda_highlights.csv').head(12)

,dimension,segment_value,customers,responders,response_rate
0,Gender,Male,206089,28525,0.138411
1,Gender,Female,175020,18185,0.103902
2,Previously_Insured,0,206481,46552,0.225454
3,Previously_Insured,1,174628,158,0.000905
4,Vehicle_Age,> 2 Years,16007,4702,0.293746
5,Vehicle_Age,1-2 Year,200316,34806,0.173755
6,Vehicle_Age,< 1 Year,164786,7202,0.043705
7,Vehicle_Damage,Yes,192413,45728,0.237655
8,Vehicle_Damage,No,188696,982,0.005204
9,age_group,36-45,70532,15192,0.215392
